# Fine Tuning SAM Audio

In [1]:
from anatolian_sam.model_utils import load_sam_audio
from anatolian_sam.latents_dataloader import (
    TurkishMusicLatentsDataset,
    get_train_val_loaders,
)
from anatolian_sam.flow_utils import expand_to_256

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.11.0+cu130)
    Python  3.10.19 (you have 3.11.15)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available

## Load SAM Audio and Processor

In [2]:
base_model, processor = load_sam_audio()

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 35128.17it/s]


/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Some weights of RobertaModel were not initialized from t

## Setup PEFT/LoRA Fine-Tune

In [3]:
import torch

base_model.text_encoder.requires_grad_(False)

with torch.no_grad():
    text_features, text_mask = base_model.text_encoder(["zurna"])

print("Text embeddings shape:", text_features.shape)

Text embeddings shape: torch.Size([1, 4, 768])


### Custom Dataset

In [4]:
dataset = TurkishMusicLatentsDataset(
    jsonl_path="../data/mixed_metadata.jsonl",
    data_base_path="../data/latents",
    text_encoder=base_model.text_encoder,
)

Loaded 720 audio tuples from mixed_metadata.jsonl


In [5]:
print(f"Dataset size: {len(dataset)}")
sample = dataset[0]
print("Sample keys:", sample.keys())
print(f"Instrument: {sample['target_filename']}")

Dataset size: 720
Sample keys: dict_keys(['mixture_latent', 'target_latent', 'text_features', 'text_mask', 'mixture_filename', 'target_filename'])
Instrument: target_0000_ney.wav


In [ ]:
BATCH_SIZE = 8

In [7]:
train_loader, test_loader = get_train_val_loaders(dataset, batch_size=BATCH_SIZE)

In [8]:
for batch in train_loader:
    print("Batch keys:", batch.keys())
    print("Mixture latent shape:", batch["mixture_latent"].shape)
    print("Target latent shape:", batch["target_latent"].shape)
    print("Text features shape:", batch["text_features"].shape)
    print("Text mask shape:", batch["text_mask"].shape)
    break  # Just check the first batch

Batch keys: dict_keys(['mixture_latent', 'target_latent', 'text_features', 'text_mask'])
Mixture latent shape: torch.Size([8, 128, 125])
Target latent shape: torch.Size([8, 128, 125])
Text features shape: torch.Size([8, 10, 768])
Text mask shape: torch.Size([8, 10])


### Wrap with PEFT/LoRA

In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # The developers of SAMAudio named the attention layers in their DiT (the DiT handles the actual latent generation) wq and wv
    target_modules=["wq", "wv"],
    lora_dropout=0.05,
    bias="none",
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 6,468,359,530 || trainable%: 0.0648


### Initialize Flow Matching Scheduler and Accelerate

In [10]:
NUM_EPOCHS = 40
LEARNING_RATE = 1e-4

In [11]:
from diffusers import FlowMatchEulerDiscreteScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from accelerate import Accelerator
from torch.optim import AdamW
import math

# initialize flow-matching scheduler
scheduler = FlowMatchEulerDiscreteScheduler(
    num_train_timesteps=1000,
    shift=1.0,  # standard shift value for flow matching
)

# initialize accelerator for vram and device management
accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="bf16")

optimizer = AdamW(peft_model.parameters(), lr=LEARNING_RATE)

# CRITICAL: Because we are accumulating gradients over 4 batches,
# the optimizer only takes a "step" once every 4 forward passes.
num_update_steps_per_epoch = math.ceil(
    len(train_loader) / accelerator.gradient_accumulation_steps
)
max_train_steps = NUM_EPOCHS * num_update_steps_per_epoch

# 2. Define Warmup Steps
# A standard rule of thumb for LoRA is warming up for 5% to 10% of total training steps.
num_warmup_steps = int(max_train_steps * 0.10)

# 3. Initialize the Scheduler
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps,
)

# Pass everything to accelerator.prepare
peft_model, optimizer, train_loader, test_loader, lr_scheduler = accelerator.prepare(
    peft_model, optimizer, train_loader, test_loader, lr_scheduler
)

print("Flow-Matching Scheduler and Accelerator Initialized!")
print(f"Total Training Steps: {max_train_steps}")
print(f"Warmup Steps: {num_warmup_steps}")

Flow-Matching Scheduler and Accelerator Initialized!
Total Training Steps: 720
Warmup Steps: 72


## Custom LoRA Flow-Matching Training Loop

In [12]:
import inspect

# Inspect the forward method of the underlying base model
signature = inspect.signature(base_model.forward)

print("SAM Audio Forward Signature:")
for param in signature.parameters.values():
    print(f"- {param.name}: {param.default}")

SAM Audio Forward Signature:
- noisy_audio: <class 'inspect._empty'>
- audio_features: <class 'inspect._empty'>
- text_features: <class 'inspect._empty'>
- time: <class 'inspect._empty'>
- masked_video_features: None
- text_mask: None
- anchor_ids: None
- anchor_alignment: None
- audio_pad_mask: None


In [ ]:
import wandb
import os

wandb.init(
    entity="zeerafle-sivas-cumhuriyet-university",
    project="turkish-sam-audio",
    name="lora-flow-matching-run",
    config={
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    },
    dir="../wandb",
)

# Setup Checkpointing Directory
save_directory = "../checkpoints/turkish_sam_audio_lora_best"
os.makedirs(save_directory, exist_ok=True)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [15]:
import torch
import torch.nn.functional as F
from tqdm.autonotebook import tqdm


# Ensure the frozen base model is in eval mode, but PEFT adapters are training
peft_model.train()

# Tracking Variables
best_loss = float("inf")
global_step = 0

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0  # Accumulator to calculate average loss per epoch

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS - 1}")

    # train_dataloader yields the Tuple: Mixture, Target, Text Embeddings
    for step, batch in enumerate(pbar):
        # accelerate.accumulate handles the 16-step gradient hold for your T4
        with accelerator.accumulate(peft_model):
            # 1. Unpack the Pre-Computed Tensors
            mixture_latent = batch["mixture_latent"].transpose(
                1, 2
            )  # Conditioner 1: [Batch, 125, 128]
            target_latent = batch["target_latent"].transpose(
                1, 2
            )  # Ground Truth x_1: [Batch, 125, 128]
            text_features = batch["text_features"]  # Conditioner 2: Frozen Embeddings
            text_mask = batch["text_mask"]

            # Time Sampling
            # You randomly select a continuous timestep (t) between 0 and 1[cite: 1062].
            t = torch.rand((BATCH_SIZE,), device=accelerator.device)
            t_expanded = t.view(BATCH_SIZE, 1, 1)  # Reshape for tensor broadcasting

            # additional step, Meta's trick (https://github.com/facebookresearch/sam-audio/blob/68b48d48fff1ad776d3afefbe634eb5f5d60ba7b/sam_audio/model/model.py#L184)
            mixture_256 = expand_to_256(mixture_latent)
            target_256 = expand_to_256(target_latent)

            # generate Noise and Noisy State in the 256-dimensional space
            noise_256 = torch.randn_like(target_256)
            x_t_256 = (1 - t_expanded) * noise_256 + t_expanded * target_256

            # The Forward Pass
            # Wrap the forward pass to automatically handle mixed precision casting
            with accelerator.autocast():
                # You feed the noisy state (x_t), the timestep (t), and your two completely clean conditioners (the Mixture Latent and the Text Prompt) into your peft_model[cite: 1064].
                predicted_velocity = peft_model(
                    noisy_audio=x_t_256,
                    time=t,
                    audio_features=mixture_256,
                    text_features=text_features,
                    text_mask=text_mask,
                )

                # calculate the True Vector Field
                # Flow-matching models predict the trajectory pointing from noise to target.
                true_velocity = target_256 - noise_256
                # Vector Field Discrepancy Loss
                # Calculate the MSE between the predicted trajectory and the true trajectory[cite: 1107].
                loss = F.mse_loss(predicted_velocity, true_velocity)

            # the Update
            # Push the gradients back strictly to the tiny q_proj and v_proj LoRA weights[cite: 1066].
            accelerator.backward(loss)

            # Optimizer automatically respects the gradient_accumulation_steps=16 we set earlier
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        # Update the tqdm progress bar with the current loss
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Log Step-Level Metrics to W&B
        wandb.log(
            {
                "train/step_loss": loss.item(),
                "train/learning_rate": lr_scheduler.get_last_lr()[0],
                "global_step": global_step,
            }
        )

        epoch_loss += loss.item()
        global_step += 1

    # epoch-Level Logging and Checkpointing
    avg_epoch_loss = epoch_loss / len(train_loader)
    wandb.log({"train/epoch_loss": avg_epoch_loss, "epoch": epoch})

    print(f"--- Epoch {epoch} Complete | Avg Loss: {avg_epoch_loss:.4f} ---")

    # Save Best Model Logic
    if avg_epoch_loss < best_loss:
        print(
            f"New best loss found: {avg_epoch_loss:.4f} (Previous: {best_loss:.4f}). Saving checkpoint..."
        )
        best_loss = avg_epoch_loss

        # Ensure multi-GPU operations are synced before saving
        accelerator.wait_for_everyone()

        # Only the main process handles the file I/O to prevent corruption
        if accelerator.is_main_process:
            unwrapped_model = accelerator.unwrap_model(peft_model)
            unwrapped_model.save_pretrained(save_directory, safe_serialization=True)

# Finish the W&B run cleanly
wandb.finish()

Epoch 0/39:   0%|          | 0/72 [00:00<?, ?it/s]

Epoch 0/39: 100%|██████████| 72/72 [00:28<00:00,  2.51it/s, loss=0.6212]


--- Epoch 0 Complete | Avg Loss: 0.5857 ---
New best loss found: 0.5857 (Previous: inf). Saving checkpoint...


Epoch 1/39: 100%|██████████| 72/72 [00:27<00:00,  2.64it/s, loss=0.6230]


--- Epoch 1 Complete | Avg Loss: 0.5459 ---
New best loss found: 0.5459 (Previous: 0.5857). Saving checkpoint...


...

Epoch 38/39: 100%|██████████| 72/72 [00:27<00:00,  2.66it/s, loss=0.2157]


--- Epoch 38 Complete | Avg Loss: 0.2622 ---


Epoch 39/39: 100%|██████████| 72/72 [00:27<00:00,  2.65it/s, loss=0.2655]

--- Epoch 39 Complete | Avg Loss: 0.2658 ---


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
global_step,▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
train/epoch_loss,█▇▄▄▃▃▂▂▂▂▂▂▁▂▁▂▁▁▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁
train/learning_rate,▁▂▃▆▆███████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▁▁
train/step_loss,█▇▄▆▃▃▅▄▃▂▄▂▁▆▄▂▃▁▁▃▆▃▅▂▃▄▃▅▅▄▂▅▃▂▁▅▃▆▃▃
epoch,39
global_step,2879
train/epoch_loss,0.2658
train/learning_rate,0
train/step_loss,0.26552


In [16]:
# 1. Ensure # 1. Ensure all GPUs have finished their final calculations
accelerator.wait_for_everyone()

# 2. Define a DISTINCT output directory for the final epoch
# Notice the "_final" suffix to prevent overwriting your "_best" checkpoint
save_directory = "../checkpoints/turkish_sam_audio_lora_final"
os.makedirs(save_directory, exist_ok=True)

# 3. Unwrap and Save the final epoch state
if accelerator.is_main_process:
    unwrapped_model = accelerator.unwrap_model(peft_model)
    unwrapped_model.save_pretrained(
        save_directory,
        safe_serialization=True,
    )

print(f"Final epoch LoRA adapters successfully saved to: {save_directory}")

Final epoch LoRA adapters successfully saved to: ../checkpoints/turkish_sam_audio_lora_final
